# Payment Guard - extractor 0.5B (LoRA, Unsloth)
Runtime > T4 GPU. El modelo solo EXTRAE campos; nunca decide pagos.

In [4]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [5]:
import os, getpass
HF_USER = "P1p2gamer26"
MODEL_ID = f"{HF_USER}/payment-guard-extractor-0.5b"
BASE = "unsloth/Qwen2.5-0.5B-Instruct"  # si sobra tiempo: unsloth/Qwen2.5-1.5B-Instruct
# Cambia por el raw del repo publico una vez publicado (o sube ml/train.jsonl con el panel de archivos de Colab)
TRAIN_URL = "https://raw.githubusercontent.com/P1p2gamer26/payment-guard/main/ml/train.jsonl"
if not os.path.exists("train.jsonl"):
    !wget -q -O train.jsonl $TRAIN_URL || echo "descarga fallida: sube train.jsonl a mano"
!wc -l train.jsonl

1290 train.jsonl


In [6]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], random_state=30)
ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = ds.map(lambda b: {"text": [tok.apply_chat_template(m, tokenize=False) for m in b["messages"]]}, batched=True)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=SFTConfig(
    dataset_text_field="text", max_seq_length=2048, per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="linear", warmup_ratio=0.05, logging_steps=5,
    fp16=True, optim="adamw_8bit", output_dir="out", report_to="none", seed=30))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.10.6 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1290 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,290 | Num Epochs = 2 | Total steps = 162
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,0.255267
10,0.136421
15,0.082783
20,0.081568
25,0.039450
30,0.051185
35,0.035967
40,0.032619
45,0.039510
50,0.047092


Unsloth: Restored added_tokens_decoder metadata in out/checkpoint-162/tokenizer_config.json.


TrainOutput(global_step=162, training_loss=0.042025393794900105, metrics={'train_runtime': 355.4048, 'train_samples_per_second': 7.259, 'train_steps_per_second': 0.456, 'total_flos': 3129018231774720.0, 'train_loss': 0.042025393794900105, 'epoch': 2.0})

In [7]:
# prueba rapida con una factura nueva
FastLanguageModel.for_inference(model)
import json
from unsloth import get_chat_template
msgs = json.loads(open("train.jsonl").readline())["messages"][:2]
x = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
print(tok.decode(model.generate(input_ids=x, max_new_tokens=200, do_sample=False)[0][x.shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"beneficiary_name": "Brightpath Consulting Prueba LLC", "beneficiary_id": null, "amount": "2680.00", "currency": "USD", "issue_date": "2026-12-21", "invoice_number": "0494", "bank_account": "FICT-ACH-703859", "concept": "monthly retainer - October 2026"}


In [8]:
# Token HF con permiso write: se pide aqui, NO se guarda en el cuaderno.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass.getpass("HF token (write): ")
model.push_to_hub_gguf(MODEL_ID, tok, quantization_method="q4_k_m", token=HF_TOKEN)
print("https://huggingface.co/" + MODEL_ID)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_y96rb_ch/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/tmp/unsloth_gguf_y96rb_ch`: 100%|██████████| 1/1 [00:04<00:00,  4.99s/it]


Successfully copied all 1 files from cache to `/tmp/unsloth_gguf_y96rb_ch`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:07<00:00,  7.81s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_y96rb_ch`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11696-mix-7baeb9a (app-b11696-mix-7baeb9a-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_y96rb_ch_gguf/Qwen2.5-0.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF convers

RuntimeError: Failed to upload to Hugging Face Hub: HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6ad38f0e-51b8cc321fa54d9055d65025;e2301f4f-3bc6-43d0-a689-202ef9401260)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.